In [1]:
%load_ext autoreload
%autoreload 2
# these packages cannot be autoreloaded
%aimport -modal
%aimport -modal_proto
%aimport -synchronicity

import sys, time
from pathlib import Path
sys.path.append(str(Path().resolve().parent / "src3"))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from vibrations_pipeline import process_vibrations_modal, compare_pclk_modal, volume, VOLUME_PATH, app

In [ ]:
# Run once after kernel restart — keeps Modal app alive for the whole session.
# Must use async with app.run.aio() in Jupyter; the sync app.run() fights Jupyter's event loop.
# _app_ctx holds the context manager so it isn't garbage collected (which would stop the app).
_app_ctx = app.run.aio()
await _app_ctx.__aenter__()

[modal-client] 2026-06-17T23:52:07+0300 Loop attempt for _run_app.<locals>.heartbeat failed
Traceback (most recent call last):
  File "/home/ethantu/workspace/good-vibrations/.venv/lib/python3.12/site-packages/modal/_utils/async_utils.py", line 571, in loop_coro
    await asyncio.wait_for(async_f(), timeout=timeout)
  File "/home/ethantu/.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu/lib/python3.12/asyncio/tasks.py", line 520, in wait_for
    return await fut
           ^^^^^^^^^
  File "/home/ethantu/workspace/good-vibrations/.venv/lib/python3.12/site-packages/modal/runner.py", line 66, in _heartbeat
    await client.stub.AppHeartbeat(request, retry=Retry(attempt_timeout=HEARTBEAT_TIMEOUT))
modal.exception.ConflictError: App state is APP_STATE_STOPPED
[modal-client] 2026-06-17T23:52:22+0300 Loop attempt for _run_app.<locals>.heartbeat failed
Traceback (most recent call last):
  File "/home/ethantu/workspace/good-vibrations/.venv/lib/python3.12/site-packages/modal/_utils/async

In [4]:
BASE_SAMPLE_DIR = Path('/home/ethantu/workspace/good-vibrations/experiment-20/data/samples')
sample_dir = BASE_SAMPLE_DIR / '000000'

In [5]:
# Correctness check: compare sequential vs batched vs batched_optimized on 3 ROIs.
# Re-run this cell any time pclk.py changes to catch regressions before benchmarking.
compare_pclk_modal.remote(sample_dir.name, n_rois=3, batch_size=1024)

{'seq_vs_batched': np.True_,
 'seq_vs_opt': np.True_,
 'batched_vs_opt': np.True_}

In [6]:
# upload — force=True overwrites if already present
t0 = time.perf_counter()
with volume.batch_upload(force=True) as batch:
    batch.put_file(sample_dir / 'inputs/00_raw_vibrations.npy', f"{sample_dir.name}/inputs/00_raw_vibrations.npy")
    batch.put_file(sample_dir / 'metadata.jsonl',               f"{sample_dir.name}/metadata.jsonl")
t_upload = time.perf_counter() - t0
print(f"upload: {t_upload:.1f}s")

upload: 2.0s


In [7]:
# memory debug run — bs=2048 to stress-test, verbose=2 prints mem at each stage
fn = process_vibrations_modal.with_options(env={"PCLK_MODE": "batched_optimized"})
t0 = time.perf_counter()
fn.remote(sample_dir.name, pclk_batch_size=2048, verbose=2)
t_modal = time.perf_counter() - t0
print(f"batched_optimized bs=2048 total: {t_modal:.1f}s")

ExecutionError: Could not deserialize remote exception due to local error:
Deserialization failed because the 'cupy' module is not available in the local environment.
This can happen if your local environment does not have the remote exception definitions.
Here is the remote traceback:
Traceback (most recent call last):
  File "/pkg/modal/_runtime/container_io_manager.py", line 915, in handle_input_exception
    yield
  File "/pkg/modal/_container_entrypoint.py", line 189, in run_input_sync
    values = io_context.call_function_sync()
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/pkg/modal/_runtime/container_io_manager.py", line 218, in call_function_sync
    expected_value_or_values = self.finalized_function.callable(*args, **kwargs)
                               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/vibrations_pipeline.py", line 263, in process_vibrations_modal
    process_vibrations(VOLUME_PATH / sample_dir_name, **kwargs)
  File "/root/vibrations_

In [ ]:
# warm run — true compute time
fn = process_vibrations_modal.with_options(env={"PCLK_MODE": "batched_optimized"})
t0 = time.perf_counter()
fn.remote(sample_dir.name, pclk_batch_size=2048, verbose=2)
t_modal = time.perf_counter() - t0
print(f"batched_optimized bs=2048 total: {t_modal:.1f}s")

In [ ]:
# 3. download outputs back to local
ESSENTIAL_FILES = [
    "inputs/01_raw_shifts.npy",
    "inputs/03_fft_shifts.npz",
    "inputs/04_recovered_audio.wav",
    "inputs/05_processed_fft.npy",
]
SYMLINKS = [
    ("recovered_audio.wav", "inputs/04_recovered_audio.wav"),
    ("X.npy",               "inputs/05_processed_fft.npy"),
]

t0 = time.perf_counter()
for rel in ESSENTIAL_FILES:
    local_path = sample_dir / rel
    local_path.parent.mkdir(parents=True, exist_ok=True)
    with open(local_path, "wb") as f:
        for chunk in volume.read_file(f"{sample_dir.name}/{rel}"):
            f.write(chunk)

for dst_rel, src_rel in SYMLINKS:
    dst = sample_dir / dst_rel
    src = sample_dir / src_rel
    if dst.exists() or dst.is_symlink(): dst.unlink()
    dst.symlink_to(src.relative_to(dst.parent))

t_download = time.perf_counter() - t0
print(f"download: {t_download:.1f}s")

download: 4.6s


In [ ]:
print(f"\n--- full pipeline summary (batched pclk) ---")
print(f"upload:                {t_upload:.1f}s")
print(f"modal (pclk+pipeline): {t_modal:.1f}s")
print(f"download:              {t_download:.1f}s")
print(f"total:                 {t_upload + t_modal + t_download:.1f}s")


--- full pipeline summary (batched pclk) ---
upload:                2.0s
modal (pclk+pipeline): 34.7s
download:              4.6s
total:                 41.3s
